# 행동 분석 (Action Recognition)

보안카메라 앵글로 촬영된 짧은 클립들을 이용해 **정상 행동 vs 이상행동**을
VLM이 구분할 수 있는지 실습합니다.

## 사전 준비
```bash
uv sync
ollama pull qwen3-vl:4b
```

In [ ]:
import video_utils as vu

NORMAL = "data/3_cctv_normal_camera_rotate.mp4"
ANOMALY_1 = "data/3_cctv_anomaly_two_thieves.mp4"
ANOMALY_2 = "data/3_cctv_anomaly_thief_creep.mp4"

for path in [NORMAL, ANOMALY_1, ANOMALY_2]:
    print(path, vu.get_video_info(path))

## 판단 기준을 프롬프트에 명시
"이상하면 알려줘"처럼 모호하게 묻지 않고, 무엇이 정상이고 무엇이 이상행동인지
범주를 먼저 정의해줍니다 (2강 Structured Output 원칙과 동일).

In [ ]:
ACTION_PROMPT = """
아래는 보안카메라에서 추출한 프레임들이야. 다음 JSON 스키마로만 답해.
{
  "is_anomaly": boolean,
  "category": "normal" | "intrusion" | "loitering" | "other",
  "reason": "string"
}
정상 범주: 카메라 점검, 단순 이동/모니터링, 아무 일도 일어나지 않음.
이상행동 범주: 침입(문/창문을 몰래 통과), 낯선 사람이 서성임(loitering),
카메라를 의식하며 숨거나 피하는 행동.
확인되지 않으면 추측하지 말고 category는 "other", is_anomaly는 false로 답해.
"""


def analyze(path, n_frames=5):
    frames = vu.extract_frames_uniform(path, n_frames=n_frames)
    result = vu.ask_video_json(frames, ACTION_PROMPT)
    return frames, result

## 클립 1: 정상 (카메라 회전/모니터링)

In [ ]:
frames_normal, result_normal = analyze(NORMAL)
vu.show_frames(frames_normal)
result_normal

## 클립 2: 이상행동 (침입 정황)

In [ ]:
frames_anomaly1, result_anomaly1 = analyze(ANOMALY_1)
vu.show_frames(frames_anomaly1)
result_anomaly1

## 클립 3: 이상행동 (카메라를 의식하는 행동)

In [ ]:
frames_anomaly2, result_anomaly2 = analyze(ANOMALY_2)
vu.show_frames(frames_anomaly2)
result_anomaly2

## 결과 정리

In [ ]:
rows = [
    ("normal", result_normal),
    ("anomaly_1", result_anomaly1),
    ("anomaly_2", result_anomaly2),
]
print(f"{'클립':<12}{'is_anomaly':<12}{'category':<12}reason")
for name, r in rows:
    print(f"{name:<12}{str(r.get('is_anomaly')):<12}{r.get('category', ''):<12}{r.get('reason', '')}")

## 한계
- 프레임 수(5장)가 적으면 순간적인 행동을 놓치거나, 반대로 없는 사건을 있다고
  판단(환각)할 위험이 커집니다. 실무에서는 1차로 낮은 프레임 수로 스크리닝하고,
  `is_anomaly=true`인 구간만 프레임 수를 늘려 재확인하는 2단계 파이프라인이 효율적입니다.
- 오디오(경보음, 비명 등)는 프레임에 담기지 않으므로 이 방식만으로는 알 수 없습니다.